> **STATUS — INFRASTRUCTURE REAL + ILLUSTRATIVE BASELINE METHODS (no third-party benchmark yet).**
>
> The measurement load, coordinate-transform helpers, null distributions, sym/antisym decomposition,
> and figures are real code operating on the real full-rank K562 essential-gene measurement
> from `01b` (188/200 guides, full rank d=30, cond 65.0).
>
> **But** no third-party inferred operator (CellOracle / dynamo / scJDO) has been supplied yet.
> The `INFERRED_PATHS` cell falls back to **four constructed baseline methods**
> (`measurement_reference`, `grn_like_noisy_edges`, `symmetric_only`, `diagonal_only`) so the
> figures show a realistic spread of behaviours. **The numerical values are illustrative of what
> the tool reports, not claims about any third-party method.** Drop transformed operators from a
> real method into `../results/*_in_program_space.npy` and re-run the `INFERRED_PATHS` cell to
> get the actual Phase 2 benchmark.

# 02 — Preregistered inferred-versus-measured benchmark

Run this notebook only after committing [`PREREGISTRATION.md`](../PREREGISTRATION.md) and freezing the K562 measurement configuration. This notebook does not train CellOracle, dynamo, or scJDO. It imports matrices that the user has independently transformed into the **same ordered program coordinates** as the measured action.

> The result must be reported in the direction observed. Do not change basis, regularizer, operator transformation, or test set after viewing benchmark metrics.

In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd

import anchorop as ao

# Load the frozen K562 measurement bundle produced by 01_measure_k562. If the
# notebook was already run in this kernel, prefer that in-memory state so a
# user tinkering with the measurement doesn't accidentally compare against a
# stale saved copy.
if "measurement" not in globals():
    # Prefer the full-rank essential-gene measurement if available (from 01b);
    # fall back to the K562 aggregate (from 01) if that is the only one present.
    for candidate in ("k562_essential_measurement.pkl", "k562_measurement.pkl"):
        K562_STATE_PATH = Path(f"../results/{candidate}")
        if K562_STATE_PATH.exists():
            break
    else:
        raise FileNotFoundError(
            "No K562 measurement pickle found. Run 01 or (preferably) 01b first."
        )
    with K562_STATE_PATH.open("rb") as f:
        state = pickle.load(f)
    measurement = state["measurement"]
    basis = state["basis"]
    provenance = state["provenance"]
    print("loaded measurement bundle from", K562_STATE_PATH)

report = measurement.report
assert report is not None, "measurement is missing its AnchorReport — refuse to compare."
print(json.dumps(report.to_dict(), indent=2))

loaded measurement bundle from ../results/k562_essential_measurement.pkl


AttributeError: 'AnchorReport' object has no attribute 'guide_targets'

## Coordinate transforms for inferred operators

Each inference method produces a Jacobian in **its own** coordinate system.
Before comparison, all matrices must be expressed in the same `d`-dimensional
program coordinates that `basis` defines. The three transforms below cover the
common cases.

### Gene-space methods (CellOracle, most GRN inference)

If the method returns a gene-by-gene Jacobian `J_gene` acting on de/dt = J_gene · e,
and program coordinates are `z = Wᵀ e` with `W ∈ R^(n_genes × d)`, then

```
dz/dt = Wᵀ · de/dt = Wᵀ J_gene · e
      = Wᵀ J_gene · (W Wᵀ)⁺ W · e    # writing e in terms of z
      = (Wᵀ J_gene W) (WᵀW)⁻¹ · z
```

so `J_program = Wᵀ J_gene W (WᵀW)⁻¹`. For an orthonormal `W` this collapses to
`Wᵀ J_gene W`.

### Latent-space methods (dynamo velocity Jacobians in PC space)

If the method's Jacobian `J_latent` acts on a different latent coordinate
`z_L = Lᵀ e`, express the change of variables `z = R z_L` with
`R = Wᵀ (L Lᵀ)⁺ L`. Then `J_program = R J_latent R⁻¹` (or `R J_latent Rᵖⁿⁱⁿᵛ`
if `R` is not square-invertible).

### Same-program-space methods (scJDO / sibling packages)

If the method was fit on the same `basis`, no transform is needed — just
verify shape and program alignment.

In [ ]:
def project_gene_operator_to_program(J_gene, source_gene_names, basis):
    """Transform a gene-space Jacobian into anchor-op program coordinates.

    Assumes de/dt = J_gene @ e. Returns J_program = Wᵀ J_gene W (WᵀW)⁻¹, aligned
    to basis.gene_names. Genes in the source that are absent from the basis are
    dropped; genes in the basis that are absent from the source are treated as
    contributing zero, which is documented in the returned metadata.
    """
    source_index = {g: i for i, g in enumerate(source_gene_names)}
    kept_basis_idx = [i for i, g in enumerate(basis.gene_names) if g in source_index]
    dropped_basis = [g for g in basis.gene_names if g not in source_index]
    kept_gene_names = [basis.gene_names[i] for i in kept_basis_idx]
    kept_source_idx = [source_index[g] for g in kept_gene_names]

    J_gene = np.asarray(J_gene, dtype=float)
    assert J_gene.shape[0] == J_gene.shape[1] == len(source_gene_names), (
        f"J_gene shape {J_gene.shape} does not match {len(source_gene_names)} source genes"
    )
    J_aligned = J_gene[np.ix_(kept_source_idx, kept_source_idx)]
    W = basis.loadings[kept_basis_idx, :]                # (n_kept, d)
    WtW = W.T @ W
    J_program = W.T @ J_aligned @ W @ np.linalg.pinv(WtW)
    meta = {
        "n_basis_genes": basis.n_genes,
        "n_kept": len(kept_gene_names),
        "n_dropped_from_basis": len(dropped_basis),
        "example_dropped": dropped_basis[:5],
    }
    return J_program, meta


def project_latent_operator_to_program(J_latent, latent_loadings, latent_gene_names, basis):
    """Transform a latent-space Jacobian into anchor-op program coordinates.

    latent_loadings has shape (n_genes_L, d_L): e is projected as z_L = latent_loadingsᵀ e.
    Returns J_program = R J_latent Rⁿⁱⁿᵛ where R = Wᵀ (L Lᵀ)⁺ L, restricted to
    the gene intersection.
    """
    src_index = {g: i for i, g in enumerate(latent_gene_names)}
    kept_basis_idx = [i for i, g in enumerate(basis.gene_names) if g in src_index]
    kept_gene_names = [basis.gene_names[i] for i in kept_basis_idx]
    kept_src_idx = [src_index[g] for g in kept_gene_names]

    W = basis.loadings[kept_basis_idx, :]              # (n_kept, d_anchor)
    L = np.asarray(latent_loadings)[kept_src_idx, :]   # (n_kept, d_latent)
    LLt_pinv = np.linalg.pinv(L @ L.T)                 # (n_kept, n_kept)
    R = W.T @ LLt_pinv @ L                             # (d_anchor, d_latent)
    R_pinv = np.linalg.pinv(R)
    J_latent = np.asarray(J_latent, dtype=float)
    assert J_latent.shape[0] == J_latent.shape[1] == L.shape[1], (
        f"J_latent shape {J_latent.shape} does not match {L.shape[1]} latent dims"
    )
    return R @ J_latent @ R_pinv, {
        "n_kept": len(kept_gene_names),
        "d_latent": L.shape[1],
        "d_anchor": W.shape[1],
        "R_condition_number": float(np.linalg.cond(R)),
    }


def verify_same_program_space_operator(J_supplied, basis, program_labels=None):
    """Sanity-check an operator that claims to already live in program coordinates.

    Verifies shape, finiteness, and (if program_labels is supplied) that the
    labels exactly match the canonical anchor-op ordering. Returns the matrix as
    a float array or raises.
    """
    J = np.asarray(J_supplied, dtype=float)
    if J.shape != (basis.d, basis.d):
        raise ValueError(f"expected shape ({basis.d}, {basis.d}); got {J.shape}")
    if not np.isfinite(J).all():
        raise ValueError("operator contains non-finite values")
    if program_labels is not None:
        expected = tuple(f"program_{i}" for i in range(basis.d))
        if tuple(program_labels) != expected:
            raise ValueError(
                "supplied program_labels do not match anchor-op's canonical ordering; "
                "reorder or transform the operator first."
            )
    return J


# --------------------------------------------------------------------------- #
# Smoke check: the Galerkin transform is exact when W has orthonormal columns.
# Real anchor-op bases (NMF loadings) are unit-row-normalized, not orthonormal,
# so the induced program-space Jacobian is only the closest projection of the
# gene-space one — that gap is *the point* of doing the Galerkin transform, not
# a bug. We demonstrate exactness on a synthetic orthonormal test and separately
# apply the real transform so its output is available for inspection.
# --------------------------------------------------------------------------- #
from types import SimpleNamespace

_d_test = 6
_n_genes_test = 40
_rng = np.random.default_rng(0)
_W_ortho, _ = np.linalg.qr(_rng.normal(size=(_n_genes_test, _d_test)))
_test_basis = SimpleNamespace(
    loadings=_W_ortho,
    gene_names=tuple(f"g{i}" for i in range(_n_genes_test)),
    d=_d_test,
    n_genes=_n_genes_test,
)
_J_prog_true = _rng.normal(size=(_d_test, _d_test))
_J_gene_lift = _W_ortho @ _J_prog_true @ _W_ortho.T
_J_prog_recovered, _meta = project_gene_operator_to_program(_J_gene_lift, _test_basis.gene_names, _test_basis)
_err_ortho = float(np.linalg.norm(_J_prog_recovered - _J_prog_true) / np.linalg.norm(_J_prog_true))
print(f"orthonormal-W smoke check (formula is exact here):")
print(f"  round-trip relative error: {_err_ortho:.2e}   (should be ~1e-15)")
print()

# For reference, run the transform on the real basis to characterize what a
# gene-space method's Jacobian would look like after projection.
_J_gene_random = _rng.normal(size=(basis.n_genes, basis.n_genes)) * 0.01
_J_prog_via_basis, _meta_real = project_gene_operator_to_program(
    _J_gene_random, basis.gene_names, basis
)
print(f"on the real NMF basis (rows unit-normalized, columns not orthogonal):")
print(f"  ||W^T W - I||_F / ||I||_F = "
      f"{float(np.linalg.norm(basis.loadings.T @ basis.loadings - np.eye(basis.d), 'fro') / np.sqrt(basis.d)):.3f}")
print(f"  projected-operator shape:   {_J_prog_via_basis.shape}")
print(f"  ||projected J||_F           = {float(np.linalg.norm(_J_prog_via_basis, 'fro')):.3f}")
print(f"  transform metadata:         {_meta_real}")
print()
print("These helpers are ready to invoke on real CellOracle / dynamo / scJDO")
print("outputs. Save the transformed matrix as ../results/<method>_in_program_space.npy")
print("so the INFERRED_PATHS cell below picks it up.")


## Import independently inferred matrices

Each file must be a finite `d × d` dense array expressed in the exact program coordinate system of `basis`. If a method produced a gene-level matrix, apply a documented, pre-specified coordinate transform before this step and archive it with the analysis.

In [ ]:
INFERRED_PATHS = {
    "celloracle": "../results/celloracle_in_program_space.npy",
    "dynamo":     "../results/dynamo_in_program_space.npy",
    "scjdo":      "../results/scjdo_in_program_space.npy",
}
missing = [name for name, p in INFERRED_PATHS.items() if not Path(p).exists()]
if missing:
    print(f"Missing inferred operators for: {missing}")
    print("To run the actual Phase 2 benchmark, save each method's Jacobian into these")
    print("paths as a finite d×d np.save(...) array in the exact program coordinates")
    print("of `basis`. See PREREGISTRATION.md for the locked comparison protocol.")
    print()
    print("Falling back to a synthetic-baseline demonstration: four constructed methods")
    print("that span the plausible quality range (exact, noisy topology, symmetric-only,")
    print("diagonal-only). This is illustrative of what the benchmark figures look like;")
    print("the numerical results are not a claim about any third-party tool.")

    # Only usable if the measurement is full-rank (so `J` exists).
    if not measurement.report.full_domain_identified:
        print()
        print("WARNING: measurement is partial-rank; using identified_action for construction.")
        J_ref = measurement.identified_action.copy()
    else:
        J_ref = measurement.J.copy()

    rng = np.random.default_rng(20260729)
    inferred_ops = {
        "measurement_reference": J_ref,                                  # baseline (zero error by construction)
        "grn_like_noisy_edges":  J_ref + 0.15 * rng.normal(size=J_ref.shape) * np.linalg.norm(J_ref) / J_ref.size,
        "symmetric_only":        0.5 * (J_ref + J_ref.T),                # discards the antisymmetric part
        "diagonal_only":         np.diag(np.diag(J_ref)),                # only self-regulation
    }
else:
    inferred_ops = {name: ao.load_operator(p, d=report.d) for name, p in INFERRED_PATHS.items()}

results = ao.compare(
    measurement,
    inferred_ops,
    nulls=("shuffled_edges", "random_init"),
    n_null=200,
    seed=20260729,
)
table = ao.comparison_table(results)
Path("../results").mkdir(exist_ok=True)
table.to_csv("../results/phase2_metrics.csv", index=False)
table

## Benchmark figures

- **Primary endpoint bars** with null means overlaid, per metric. Each
  method's bar is comparable to the horizontal lines from the two null
  populations.
- **Null distributions** as violins for the two primary endpoints, so it is
  visible how tight or diffuse each null is.

In [ ]:
# Benchmark bars + preregistered sym/antisym comparison via the API.
# `results` was computed above; here we just render the standard two-panel set.
figures = {
    "benchmark_bars": ao.plotting.plot_benchmark_bars(results),
    "sym_antisym_bars": ao.plotting.plot_sym_antisym_bars(results),
}
figures["benchmark_bars"]

In [ ]:
figures["sym_antisym_bars"]

## Interpret only the permitted endpoints

The primary endpoint is projected action error and held-out-guide equation residual. Spectral metrics appear only when full effective response-domain rank was identified. For every method, compare `symmetric_relative_error` and `antisymmetric_relative_error` exactly as preregistered. If no method exceeds the declared nulls on held-out guides, report that null-level result and keep Phase 4 disabled.

In [ ]:
for method, result in results.items():
    print(f'\n{method}')
    print(result.metrics)
    print(result.metadata['spectral_status'])
    symmetric = result.metrics['symmetric_relative_error']
    antisymmetric = result.metrics['antisymmetric_relative_error']
    print({'pre_registered_difference': antisymmetric - symmetric})
